In [ ]:
import pandas as pd
#disply
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
# 2. Simplified ID Function (Matching your extraction logic)
def create_hhid(row):
    # Convert to numeric first to handle potential floats, then to int, then to string
    fsu = str(int(float(row['FSU_Serial_No'])))
    ssn = str(int(float(row['Second_Stage_Stratum_No'])))
    shn = str(int(float(row['Sample_Household_No'])))
    return f"{fsu}_{ssn}_{shn}"

In [ ]:
master_df = pd.read_csv('/content/master_file_merger_new.csv')

In [ ]:
#read data
master_df.head()

,common_id,State,Sector,District,Multiplier,hh_size,head_age,head_gender,head_marital_status,head_education_level,is_digital_hh,Household_Type,Total_Area_Land_Owned_Acres,Ration_Card_Type,Energy_Source_Cooking,food_spend_x,intoxicant_spend_x,food_spend_y,intoxicant_spend_y,food_spend,intoxicant_spend
0,46667_2_1,1,1,6,16669,5,54,1,2,6,1.0,2,1.87,3,2,41536.0,8100.0,41536.0,8100.0,41536.0,8100.0
1,46667_3_1,1,1,6,24840,4,60,1,2,12,1.0,4,0.51,3,2,10480.0,5600.0,10480.0,5600.0,10480.0,5600.0
2,46667_3_2,1,1,6,24840,5,37,1,2,4,2.0,2,0.52,2,2,103560.0,4000.0,103560.0,4000.0,103560.0,4000.0
3,46667_3_3,1,1,6,24840,5,34,1,2,6,1.0,4,0.03,3,2,77480.0,7000.0,77480.0,7000.0,77480.0,7000.0
4,46667_3_4,1,1,6,24840,3,85,1,2,13,2.0,9,0.51,3,2,51690.0,1000.0,51690.0,1000.0,51690.0,1000.0


In [ ]:
# --- 1. Load and Clean Level 11 ---
l11 = pd.read_csv('/content/LEVEL - 11 (Section 4_3).csv')

# Use the standardized 3-column ID logic
l11['common_id'] = l11.apply(create_hhid, axis=1)

# --- 2. Feature Engineering: Scores ---
# We use pd.to_numeric and mapping to ensure 1 (Yes) becomes 1 and 2 (No) becomes 0.

# Modernity: Online Shopping Habits
online_cols = ['Online_Clothing', 'Online_Footwear', 'Online_Furniture', 'Online_Mobile',
               'Online_PersonalGoods', 'Online_RecreationGoods', 'Online_HouseholdAppliances']
l11['online_score'] = l11[online_cols].apply(lambda x: pd.to_numeric(x, errors='coerce')).replace({2:0}).sum(axis=1)

# Wealth: Physical Asset Possession
possession_cols = ['Possess_Television', 'Possess_Laptop', 'Possess_Mobile',
                   'Possess_Car', 'Possess_Refrigerator', 'Possess_WashingMachine']
l11['asset_score'] = l11[possession_cols].apply(lambda x: pd.to_numeric(x, errors='coerce')).replace({2:0}).sum(axis=1)

# Socio-Economic: Welfare Dependency
free_cols = ['Free_Laptop', 'Free_Mobile', 'Free_Bicycle', 'Free_Scooter']
l11['welfare_benefit_count'] = l11[free_cols].apply(lambda x: pd.to_numeric(x, errors='coerce')).replace({2:0}).sum(axis=1)

# --- 3. Aggregate to Household Level ---
# We use .max() because these attributes are typically recorded at the household level
l11_final = l11.groupby('common_id')[['online_score', 'asset_score', 'welfare_benefit_count']].max().reset_index()

# --- 4. The Clean Merge ---
# Ensuring string consistency to avoid "Zero" match errors
master_df['common_id'] = master_df['common_id'].astype(str)
l11_final['common_id'] = l11_final['common_id'].astype(str)

# Identify only the new columns to avoid merge duplicates
new_l11_cols = [c for c in l11_final.columns if c not in master_df.columns or c == 'common_id']

master_df = master_df.merge(l11_final[new_l11_cols], on='common_id', how='left').fillna(0)

print("Level 11 Lifestyle & Wealth scores merged successfully.")
print(master_df[['common_id', 'online_score', 'asset_score']].head())

Level 11 Lifestyle & Wealth scores merged successfully.
   common_id  online_score  asset_score
0  46667_2_1           1.0          5.0
1  46667_3_1           0.0          6.0
2  46667_3_2           0.0          3.0
3  46667_3_3           2.0          5.0
4  46667_3_4           0.0          3.0


In [ ]:
#read data
master_df.head()

,common_id,State,Sector,District,Multiplier,hh_size,head_age,head_gender,head_marital_status,head_education_level,is_digital_hh,Household_Type,Total_Area_Land_Owned_Acres,Ration_Card_Type,Energy_Source_Cooking,food_spend_x,intoxicant_spend_x,food_spend_y,intoxicant_spend_y,food_spend,intoxicant_spend,online_score,asset_score,welfare_benefit_count
0,46667_2_1,1,1,6,16669,5,54,1,2,6,1.0,2,1.87,3,2,41536.0,8100.0,41536.0,8100.0,41536.0,8100.0,1.0,5.0,0.0
1,46667_3_1,1,1,6,24840,4,60,1,2,12,1.0,4,0.51,3,2,10480.0,5600.0,10480.0,5600.0,10480.0,5600.0,0.0,6.0,0.0
2,46667_3_2,1,1,6,24840,5,37,1,2,4,2.0,2,0.52,2,2,103560.0,4000.0,103560.0,4000.0,103560.0,4000.0,0.0,3.0,0.0
3,46667_3_3,1,1,6,24840,5,34,1,2,6,1.0,4,0.03,3,2,77480.0,7000.0,77480.0,7000.0,77480.0,7000.0,2.0,5.0,0.0
4,46667_3_4,1,1,6,24840,3,85,1,2,13,2.0,9,0.51,3,2,51690.0,1000.0,51690.0,1000.0,51690.0,1000.0,0.0,3.0,0.0


In [ ]:
# --- 1. Load and Prepare Level 12 ---
l12 = pd.read_csv('/content/LEVEL - 12 (Section 13).csv')

# Use the standardized 3-column ID logic for a perfect match
l12['common_id'] = l12.apply(create_hhid, axis=1)

# --- 2. Extract ONLY requested Features (Total Value and Unique Count) ---
l12_metrics = l12.groupby('common_id').agg(
    total_l12_basket_value=('VALUE', 'sum'),
    total_l12_unique_items=('ITEM_CODE', 'nunique')
).reset_index()

# --- 3. Clean Merge to Master File ---
# Force string consistency for the join
master_df['common_id'] = master_df['common_id'].astype(str)
l12_metrics['common_id'] = l12_metrics['common_id'].astype(str)

# Identify new columns to avoid duplicate column issues
new_cols = [c for c in l12_metrics.columns if c not in master_df.columns or c == 'common_id']

# Perform the merge and fill unmatched households with 0
master_df = master_df.merge(l12_metrics[new_cols], on='common_id', how='left').fillna(0)

print("Level 12 (Total Basket & Unique Items) merged successfully.")
print(master_df[['common_id', 'total_l12_basket_value', 'total_l12_unique_items']].head())

Level 12 (Total Basket & Unique Items) merged successfully.
   common_id  total_l12_basket_value  total_l12_unique_items
0  46667_2_1                 49200.0                    16.0
1  46667_3_1                 42060.0                    16.0
2  46667_3_2                 35140.0                    16.0
3  46667_3_3                 53600.0                    18.0
4  46667_3_4                 14460.0                    10.0


In [ ]:
master_df.head()

,common_id,State,Sector,District,Multiplier,hh_size,head_age,head_gender,head_marital_status,head_education_level,is_digital_hh,Household_Type,Total_Area_Land_Owned_Acres,Ration_Card_Type,Energy_Source_Cooking,food_spend_x,intoxicant_spend_x,food_spend_y,intoxicant_spend_y,food_spend,intoxicant_spend,online_score,asset_score,welfare_benefit_count,total_l12_basket_value,total_l12_unique_items
0,46667_2_1,1,1,6,16669,5,54,1,2,6,1.0,2,1.87,3,2,41536.0,8100.0,41536.0,8100.0,41536.0,8100.0,1.0,5.0,0.0,49200.0,16.0
1,46667_3_1,1,1,6,24840,4,60,1,2,12,1.0,4,0.51,3,2,10480.0,5600.0,10480.0,5600.0,10480.0,5600.0,0.0,6.0,0.0,42060.0,16.0
2,46667_3_2,1,1,6,24840,5,37,1,2,4,2.0,2,0.52,2,2,103560.0,4000.0,103560.0,4000.0,103560.0,4000.0,0.0,3.0,0.0,35140.0,16.0
3,46667_3_3,1,1,6,24840,5,34,1,2,6,1.0,4,0.03,3,2,77480.0,7000.0,77480.0,7000.0,77480.0,7000.0,2.0,5.0,0.0,53600.0,18.0
4,46667_3_4,1,1,6,24840,3,85,1,2,13,2.0,9,0.51,3,2,51690.0,1000.0,51690.0,1000.0,51690.0,1000.0,0.0,3.0,0.0,14460.0,10.0


In [ ]:
print(master_df.shape)

(261953, 26)


In [ ]:
# --- 1. Load Level 15 ---
l15 = pd.read_csv('/content/LEVEL - 15 (Section 1_1, A2,B2  C2).csv')

# Use the standardized 3-column ID logic
l15['common_id'] = l15.apply(create_hhid, axis=1).astype(str)

# --- 2. Pivot using confirmed Section codes ---
# A2 = Durables, B2 = Recurring, C2 = Misc
l15_pivoted = l15.pivot_table(
    index='common_id',
    columns='SECTION',
    values='MONTHLY_CONSUMPTION_EXP',
    aggfunc='first'
).reset_index()

# Rename to match your CLV signals
l15_pivoted = l15_pivoted.rename(columns={
    'A2': 'annual_durable_summary_L15',
    'B2': 'monthly_recurring_summary_L15',
    'C2': 'misc_services_summary_L15'
})

# --- 3. Calculate Total Expenditure ---
# Use 'L1' (Grand Total) if available; otherwise, sum the components
if 'L1' in l15_pivoted.columns:
    l15_pivoted['total_hh_exp'] = l15_pivoted['L1']
else:
    cols_to_sum = ['A1', 'A2', 'B2', 'C2']
    existing_sum_cols = [c for c in cols_to_sum if c in l15_pivoted.columns]
    l15_pivoted['total_hh_exp'] = l15_pivoted[existing_sum_cols].sum(axis=1)

# --- 4. Calculate Final Target (MPCE) ---
# We use .first() because HH_SIZE is constant for all rows of the same household
hh_size = l15.groupby('common_id')['HOUSEHOLD_SIZE'].first().reset_index()
l15_final = l15_pivoted.merge(hh_size, on='common_id', how='left')

# Calculating the target variable for your model
l15_final['target_mpce'] = l15_final['total_hh_exp'] / l15_final['HOUSEHOLD_SIZE']

# --- 5. Clean Merge to Master ---
master_df['common_id'] = master_df['common_id'].astype(str)
l15_final['common_id'] = l15_final['common_id'].astype(str)

# Identify only specific target/summary columns to avoid cluttering master_df
target_cols = ['Monthly_durable_summary_L15', 'monthly_recurring_summary_L15',
               'misc_services_summary_L15', 'target_mpce']
existing_target_cols = [c for c in target_cols if c in l15_final.columns]

master_df = master_df.merge(l15_final[['common_id'] + existing_target_cols], on='common_id', how='left').fillna(0)

print("Level 15 (Target MPCE and Expenditure Summaries) successfully merged.")
#print(master_df[['common_id', 'target_mpce']].head())

Level 15 (Target MPCE and Expenditure Summaries) successfully merged.


In [ ]:
master_df.head()

,common_id,State,Sector,District,Multiplier,hh_size,head_age,head_gender,head_marital_status,head_education_level,is_digital_hh,Household_Type,Total_Area_Land_Owned_Acres,Ration_Card_Type,Energy_Source_Cooking,food_spend_x,intoxicant_spend_x,food_spend_y,intoxicant_spend_y,food_spend,intoxicant_spend,online_score,asset_score,welfare_benefit_count,total_l12_basket_value,total_l12_unique_items,annual_durable_summary_L15,monthly_recurring_summary_L15_x,misc_services_summary_L15_x,target_mpce_x,monthly_recurring_summary_L15_y,misc_services_summary_L15_y,target_mpce_y,monthly_recurring_summary_L15,misc_services_summary_L15,target_mpce
0,22300_1_1,34,2,2,57436,4,56,1,2,12,1.0,1,0.02,3,2,13923.0,21690.0,13923.0,21690.0,13923.0,21690.0,3.0,5.0,0.0,31570.0,20.0,40000.0,39000.0,40000.0,0.0,39000.0,40000.0,0.0,39000.0,40000.0,0.0
1,22300_2_1,34,2,2,27497,4,48,1,2,5,1.0,1,0.01,3,2,10894.0,29700.0,10894.0,29700.0,10894.0,29700.0,2.0,5.0,0.0,63520.0,18.0,28000.0,32000.0,35000.0,0.0,32000.0,35000.0,0.0,32000.0,35000.0,0.0
2,22300_2_2,34,2,2,27497,2,63,1,2,12,1.0,9,0.01,3,2,11754.0,15090.0,11754.0,15090.0,11754.0,15090.0,0.0,5.0,0.0,27820.0,15.0,17500.0,17000.0,16500.0,0.0,17000.0,16500.0,0.0,17000.0,16500.0,0.0
3,22300_2_3,34,2,2,27497,2,68,1,2,5,2.0,9,0.03,3,2,13144.0,15020.0,13144.0,15020.0,13144.0,15020.0,0.0,5.0,0.0,30940.0,14.0,18000.0,17500.0,18000.0,0.0,17500.0,18000.0,0.0,17500.0,18000.0,0.0
4,22300_2_4,34,2,2,27497,2,53,1,2,13,1.0,2,0.02,3,2,11929.0,23720.0,11929.0,23720.0,11929.0,23720.0,1.0,5.0,0.0,65980.0,19.0,25000.0,24500.0,24000.0,0.0,24500.0,24000.0,0.0,24500.0,24000.0,0.0


In [ ]:
#resetting our data from common id as ascending
master_df.sort_values(by='common_id', ascending=True, inplace=True)
master_df.head()

,common_id,State,Sector,District,Multiplier,hh_size,head_age,head_gender,head_marital_status,head_education_level,is_digital_hh,Household_Type,Total_Area_Land_Owned_Acres,Ration_Card_Type,Energy_Source_Cooking,food_spend_x,intoxicant_spend_x,food_spend_y,intoxicant_spend_y,food_spend,intoxicant_spend,online_score,asset_score,welfare_benefit_count,total_l12_basket_value,total_l12_unique_items,annual_durable_summary_L15,monthly_recurring_summary_L15_x,misc_services_summary_L15_x,target_mpce_x,monthly_recurring_summary_L15_y,misc_services_summary_L15_y,target_mpce_y,monthly_recurring_summary_L15,misc_services_summary_L15,target_mpce
0,22300_1_1,34,2,2,57436,4,56,1,2,12,1.0,1,0.02,3,2,13923.0,21690.0,13923.0,21690.0,13923.0,21690.0,3.0,5.0,0.0,31570.0,20.0,40000.0,39000.0,40000.0,0.0,39000.0,40000.0,0.0,39000.0,40000.0,0.0
1,22300_2_1,34,2,2,27497,4,48,1,2,5,1.0,1,0.01,3,2,10894.0,29700.0,10894.0,29700.0,10894.0,29700.0,2.0,5.0,0.0,63520.0,18.0,28000.0,32000.0,35000.0,0.0,32000.0,35000.0,0.0,32000.0,35000.0,0.0
2,22300_2_2,34,2,2,27497,2,63,1,2,12,1.0,9,0.01,3,2,11754.0,15090.0,11754.0,15090.0,11754.0,15090.0,0.0,5.0,0.0,27820.0,15.0,17500.0,17000.0,16500.0,0.0,17000.0,16500.0,0.0,17000.0,16500.0,0.0
3,22300_2_3,34,2,2,27497,2,68,1,2,5,2.0,9,0.03,3,2,13144.0,15020.0,13144.0,15020.0,13144.0,15020.0,0.0,5.0,0.0,30940.0,14.0,18000.0,17500.0,18000.0,0.0,17500.0,18000.0,0.0,17500.0,18000.0,0.0
4,22300_2_4,34,2,2,27497,2,53,1,2,13,1.0,2,0.02,3,2,11929.0,23720.0,11929.0,23720.0,11929.0,23720.0,1.0,5.0,0.0,65980.0,19.0,25000.0,24500.0,24000.0,0.0,24500.0,24000.0,0.0,24500.0,24000.0,0.0


In [ ]:
#exporting to csv file
master_df.to_csv('master_file_final_new_final.csv', index=False)